In [ ]:
# ============================================================
# ONE CELL (BASEHOCK, binary): 5x5 CV using BEST GO-LR + NSC params
#
# - No Optuna
# - GO-LR fit once on full X using best hyperparameters -> Pi_star
# - 5x5 CV:
#     NSC configure on train only using fixed Pi_star
#     compress train/val
#     TabPFN fit on train, eval on val
# - Report:
#     Accuracy   (mean ± std)
#     ROC-AUC    (mean ± std)
#     F1         (mean ± std)
#     Runtime
# ============================================================

import os
import gc
import time
import sys
import random
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Force cuda:0 BEFORE importing torch
# ------------------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ---- IMPORTANT: import from your file ----
from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "BASEHOCK.csv"
TARGET_COL = "label"

NSC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TABPFN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Best hyperparameters from Optuna
# ------------------------------------------------------------
BEST_PARAMS = {
    "go_metric": "kl_divergence",
    "go_num_clusters": 10,
    "go_refine_passes": 2,
    "go_direction_select": False,
    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.99,
    "nsc_gamma": 1.979119016823701,
    "nsc_beta": 0.15768070718313082,
    "nsc_Mmin": 48,
    "nsc_Mmax": 640,
    "nsc_lmin": 12,
    "assume_standardized": True,
    "tabpfn_seed": 3,
}

# ------------------------------------------------------------
# Utils
# ------------------------------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_binary_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    if len(uniq) != 2:
        raise ValueError(
            f"Expected binary target with exactly 2 classes, but found {len(uniq)} classes: {uniq}"
        )
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list[int], eps: float = 1e-12) -> torch.Tensor:
    """
    delta[t] = 1 - |corr(feature_{Pi[t]}, feature_{Pi[t+1]})|
    Returns torch.Tensor shape (m-1,) on CPU.
    """
    X = torch.from_numpy(X_tr).float()  # CPU
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

def safe_binary_auc(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).astype(float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob_pos))

def summarize_metric(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), 0.0
    return float(np.mean(vals)), float(np.std(vals, ddof=1))

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

y_raw = df[TARGET_COL].to_numpy()
X_df = df.drop(columns=[TARGET_COL])

# ensure numeric predictors
for c in X_df.columns:
    if not pd.api.types.is_numeric_dtype(X_df[c]):
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

X_df = X_df.replace([np.inf, -np.inf], np.nan)
if X_df.isnull().values.any():
    X_df = X_df.fillna(X_df.median(numeric_only=True))

X_raw = X_df.values.astype(np.float32, copy=False)
y_all, class_map = ensure_binary_contiguous(y_raw)

# global scaling, matching your earlier structure
global_scaler = StandardScaler()
X_all = global_scaler.fit_transform(X_raw).astype(np.float32, copy=False)

print(f"[DATA] {DATA_FILE} | Raw X={X_raw.shape} | Scaled X={X_all.shape} | class_map={class_map}")
print(f"[GPU ] cuda_available={torch.cuda.is_available()} | visible_gpus={torch.cuda.device_count()} | NSC_DEVICE={NSC_DEVICE} | TABPFN_DEVICE={TABPFN_DEVICE}")

# 5x5 CV
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# ------------------------------------------------------------
# Main evaluation
# ------------------------------------------------------------
def run_cv_with_best_params(params):
    seed_everything(SEED)
    cleanup_cuda()

    run_start = time.time()

    # ---------- GO-LR once on full X ----------
    go = GraphFeatureOrdering(
        num_clusters=int(params["go_num_clusters"]),
        metric=params["go_metric"],
        refine=True,
        direction_select=bool(params["go_direction_select"]),
        refine_passes=int(params["go_refine_passes"]),
    )

    try:
        Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
    except RuntimeError:
        cleanup_cuda()
        Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)

    head_cfg = TabPFN25Config(
        task_type="binary",
        num_classes=2,
        device=TABPFN_DEVICE,
        random_state=int(params["tabpfn_seed"]),
    )

    accs = []
    f1s = []
    aucs = []
    fold_times = []

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        fold_start = time.time()

        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        # ---------- NSC on train only ----------
        nsc = PIDFSegPCA(
            segmentation=params["nsc_segmentation"],
            l_min=int(params["nsc_lmin"]),
            m_rule=params["nsc_m_rule"],
            gamma=float(params["nsc_gamma"]),
            beta=float(params["nsc_beta"]),
            tau=float(params["nsc_tau"]),
            M_min=int(params["nsc_Mmin"]),
            M_max=int(params["nsc_Mmax"]),
            assume_standardized=bool(params["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None
        if params["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

        X_tr_t = torch.from_numpy(X_tr)
        X_va_t = torch.from_numpy(X_va)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(params["nsc_tau"]),
            deltas=deltas,
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy()

        # ---------- TabPFN ----------
        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)

        if P.ndim == 2 and P.shape[1] >= 2:
            prob_pos = P[:, 1]
        else:
            prob_pos = P.reshape(-1)

        acc = float(accuracy_score(y_va, pred))
        f1v = float(f1_score(y_va, pred, average="macro"))
        auc = safe_binary_auc(y_va, prob_pos)

        accs.append(acc)
        f1s.append(f1v)
        aucs.append(auc)

        fold_time = time.time() - fold_start
        fold_times.append(fold_time)

        print(
            f"Fold {fold_id:02d} | "
            f"Acc={acc:.6f} | AUC={auc:.6f} | F1={f1v:.6f} | "
            f"Time={fold_time:.2f}s"
        )

        cleanup_cuda()

    total_time = time.time() - run_start

    acc_mean, acc_std = summarize_metric(accs)
    f1_mean, f1_std = summarize_metric(f1s)
    auc_mean, auc_std = summarize_metric(aucs)

    return {
        "accs": accs,
        "f1s": f1s,
        "aucs": aucs,
        "fold_times": fold_times,
        "acc_mean": acc_mean,
        "acc_std": acc_std,
        "f1_mean": f1_mean,
        "f1_std": f1_std,
        "auc_mean": auc_mean,
        "auc_std": auc_std,
        "total_time_sec": total_time,
        "Pi_star": Pi_star,
    }

# ------------------------------------------------------------
# Run final 5x5 CV
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("USING BEST HYPERPARAMETERS")
print("=" * 70)
for k, v in BEST_PARAMS.items():
    print(f"{k}: {v}")

final_results = run_cv_with_best_params(BEST_PARAMS)

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL 5x5 CV RESULTS (NO OPTUNA)")
print("=" * 70)
print(f"Accuracy : {final_results['acc_mean']:.6f} ± {final_results['acc_std']:.6f}")
print(f"ROC-AUC  : {final_results['auc_mean']:.6f} ± {final_results['auc_std']:.6f}")
print(f"F1       : {final_results['f1_mean']:.6f} ± {final_results['f1_std']:.6f}")
print(f"Runtime  : {final_results['total_time_sec']:.2f} sec ({final_results['total_time_sec']/60:.2f} min)")

[DATA] BASEHOCK.csv | Raw X=(1993, 4862) | Scaled X=(1993, 4862) | class_map={0: 0, 1: 1}
[GPU ] cuda_available=True | visible_gpus=1 | NSC_DEVICE=cuda | TABPFN_DEVICE=cuda

USING BEST HYPERPARAMETERS
go_metric: kl_divergence
go_num_clusters: 10
go_refine_passes: 2
go_direction_select: False
nsc_segmentation: uniform
nsc_m_rule: default
nsc_tau: 0.99
nsc_gamma: 1.979119016823701
nsc_beta: 0.15768070718313082
nsc_Mmin: 48
nsc_Mmax: 640
nsc_lmin: 12
assume_standardized: True
tabpfn_seed: 3
Fold 01 | Acc=0.969925 | AUC=0.996457 | F1=0.969925 | Time=614.63s
Fold 02 | Acc=0.952381 | AUC=0.994799 | F1=0.952376 | Time=631.89s
Fold 03 | Acc=0.964912 | AUC=0.994799 | F1=0.964910 | Time=551.68s
Fold 04 | Acc=0.969849 | AUC=0.997601 | F1=0.969846 | Time=536.01s
Fold 05 | Acc=0.984925 | AUC=0.997475 | F1=0.984924 | Time=611.69s
Fold 06 | Acc=0.972431 | AUC=0.996583 | F1=0.972430 | Time=642.28s
Fold 07 | Acc=0.964912 | AUC=0.995503 | F1=0.964912 | Time=568.53s
Fold 08 | Acc=0.989975 | AUC=0.998266 